# RoBERTa Final Model Training
This notebook trains a RoBERTa model for sequence classification using a predefined set of optimal hyperparameters. The dataset is split into training (80%), validation (10%), and test (10%) sets.

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report
)

In [ ]:
import wandb
import huggingface_hub

os.environ["WANDB_PROJECT"] = "roberta_degendered_final"

# wandb.login(key="YOUR_WANDB_KEY")
# huggingface_hub.login(token="YOUR_HF_TOKEN")

In [ ]:
# Model and Hyperparameter Configuration
model_name = "roberta-base"
model_cache_path = "../scratch/cache/roberta_degendered_final"

hyperparameters = {
    "learning_rate": 2.91e-04,
    "num_train_epochs": 9,
    "per_device_train_batch_size": 32,
    "weight_decay": 0.02
}

In [ ]:
# Data Preparation (80:10:10 Split)
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# First split: 80% train, 20% temp (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Second split: 10% validation, 10% test from the temp set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)

In [ ]:
# Tokenization Function
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding=False, max_length=512)
    tokens["labels"] = example["label"]
    return tokens

# Create Hugging Face Datasets
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_dataset = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_val = val_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Metrics Computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    
    # Get classification report
    report = classification_report(labels, preds, output_dict=True, zero_division=0, target_names=['Female', 'Male'])
    
    # Flatten the report for easy logging
    metrics = {
        'accuracy': report['accuracy'],
        'macro_avg_precision': report['macro avg']['precision'],
        'macro_avg_recall': report['macro avg']['recall'],
        'macro_avg_f1': report['macro avg']['f1-score'],
        'weighted_avg_precision': report['weighted avg']['precision'],
        'weighted_avg_recall': report['weighted avg']['recall'],
        'weighted_avg_f1': report['weighted avg']['f1-score'],
        'female_precision': report['Female']['precision'],
        'female_recall': report['Female']['recall'],
        'female_f1': report['Female']['f1-score'],
        'female_support': report['Female']['support'],
        'male_precision': report['Male']['precision'],
        'male_recall': report['Male']['recall'],
        'male_f1': report['Male']['f1-score'],
        'male_support': report['Male']['support']
    }
    
    print("Confusion Matrix:\n", confusion_matrix(labels, preds))

    return metrics

In [ ]:
# Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Female", 1: "Male"},
    label2id={"Female": 0, "Male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# Training Arguments
final_model_output_dir = "../scratch/final_roberta_degendered_model"
training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=hyperparameters["per_device_train_batch_size"],
    num_train_epochs=hyperparameters["num_train_epochs"],
    learning_rate=hyperparameters["learning_rate"],
    weight_decay=hyperparameters["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_avg_f1",
    evaluation_strategy="epoch",
    save_total_limit=1,
    run_name="final_roberta_degendered_training"
)

In [ ]:
# Trainer Initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val, # Use validation set for in-training evaluation
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
# Train the Model
trainer.train()

In [ ]:
# Final Evaluation on the Test Set
print("--- Final Evaluation on Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")

print("\nFinal Test Set Evaluation Results:")
print(test_results)

In [ ]:
# Save the Final Model and Tokenizer
trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)
print(f"Final model and tokenizer saved to: {final_model_output_dir}")